In [1]:
import numpy as np
import pandas as pd

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score


from multiprocessing import Pool, cpu_count

import dask
import dask.dataframe as dd
from dask.distributed import Client
client = Client(n_workers=20, memory_limit="10GB", interface='lo')

import dask_ml.cluster as dask_cluster

from pprint import pprint

### Import Overall Dataset

In [2]:
augmented_data_path = "../data/augmented_us-counties-states_latest.csv"
#augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True)).compute()
augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))
#augmented_df = pd.read_csv(augmented_data_path)

In [3]:
augmented_df.head(5)

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,totalTestsAntigen,totalTestsPeopleAntibody,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease,metrics.testPositivityRatio,metrics.vaccinationsInitiatedRatio,metrics.vaccinationsCompletedRatio
0,1001.0,2022-01-08,Autauga,Alabama,1195.0,162.0,2022-01-08,718.0,880.428571,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.435,0.487,0.388
1,1001.0,2022-08-31,Autauga,Alabama,386.0,222.0,2022-08-31,953.0,405.571429,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.209,0.569,0.450
2,1001.0,2020-04-24,Autauga,Alabama,26.0,2.0,2020-04-24,94.0,22.714286,32.532237,...,NaN,NaN,NaN,52695.0,54.0,NaN,0.0,0.079,0.000,0.000
3,1001.0,2022-04-03,Autauga,Alabama,96.0,211.0,2022-04-03,803.0,90.285714,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.020,0.553,0.439
4,1001.0,2021-01-28,Autauga,Alabama,909.0,69.0,2021-01-28,373.0,1061.714286,32.532237,...,NaN,103530.0,NaN,2116642.0,10607.0,NaN,0.0,0.205,0.030,0.007


In [4]:
### Import HSA HRR Dataset

In [5]:
hsa_hrr_data_path = "../data/ZipHsaHrr15.csv"
hsa_hrr_df = pd.read_csv(hsa_hrr_data_path)
#(hsa_hrr_df).head(5)
ZIP_to_FIPS_path = "../data/ZIP_to_FIPS.csv"
ZIP_to_FIPS_df = pd.read_csv(ZIP_to_FIPS_path)
#ZIP_to_FIPS_df.head(5)
hhs_regions_path = "../data/hhs_regions.csv"
#hhs_regions_df = pd.read_csv(hhs_regions_path)
hhs_regions_df = dd.read_csv(hhs_regions_path, assume_missing=True)
hhs_regions_df = hhs_regions_df.drop(columns=["region", "regional_office"])
hhs_regions_df = hhs_regions_df.rename(columns={"region_number":"hhs_region"})
hhs_regions_df.head(5)

#HHS_FIPS_df = pd.merge(left=hsa_hrr_df, right=ZIP_to_FIPS_df, left_on="zipcode15", right_on="ZIP", how="inner")
#HHS_FIPS_df = HHS_FIPS_df.drop(columns=["zipcode15", "hsastate","STATE","COUNTYNAME"])
#HHS_FIPS_df.head(5)

,hhs_region,state_or_territory
0,1.0,Connecticut
1,1.0,Maine
2,1.0,Massachusetts
3,1.0,New Hampshire
4,1.0,Rhode Island


In [6]:
hhs_augmented_df=dd.merge(augmented_df,hhs_regions_df, left_on="state", right_on="state_or_territory", how="left")
hhs_augmented_df.head(5)

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease,metrics.testPositivityRatio,metrics.vaccinationsInitiatedRatio,metrics.vaccinationsCompletedRatio,hhs_region,state_or_territory
0,1001.0,2022-01-08,Autauga,Alabama,1195.0,162.0,2022-01-08,718.0,880.428571,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.435,0.487,0.388,4.0,Alabama
1,1001.0,2022-08-31,Autauga,Alabama,386.0,222.0,2022-08-31,953.0,405.571429,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.209,0.569,0.450,4.0,Alabama
2,1001.0,2020-04-24,Autauga,Alabama,26.0,2.0,2020-04-24,94.0,22.714286,32.532237,...,NaN,52695.0,54.0,NaN,0.0,0.079,0.000,0.000,4.0,Alabama
3,1001.0,2022-04-03,Autauga,Alabama,96.0,211.0,2022-04-03,803.0,90.285714,32.532237,...,NaN,NaN,NaN,NaN,NaN,0.020,0.553,0.439,4.0,Alabama
4,1001.0,2021-01-28,Autauga,Alabama,909.0,69.0,2021-01-28,373.0,1061.714286,32.532237,...,NaN,2116642.0,10607.0,NaN,0.0,0.205,0.030,0.007,4.0,Alabama


### Obtain non time-varying features to cluster upon

In [7]:
stds = hhs_augmented_df.groupby('fips').std().compute()
non_time_varying_filter = stds.select_dtypes(include=[np.number]).apply(lambda x: (x < 0.0001) | (x.isna())).all()
stds_filtered = stds.loc[:, non_time_varying_filter]

In [8]:
print("Out of {} columns in hhs_augmented_df, {} are non-time varying".format(len(hhs_augmented_df.columns),len(stds_filtered.columns)))
stds_filtered.columns
# Get rid of static time series data
non_time_varying_columns = [name for name in stds_filtered.columns if not any(date in name for date in ['2020', '2021'])]
print("Out of {} columns in hhs_augmented_df, {} are non-time varying".format(len(hhs_augmented_df.columns),len(non_time_varying_columns)))
stds_filtered[non_time_varying_columns]

Out of 422 columns in hhs_augmented_df, 211 are non-time varying
Out of 422 columns in hhs_augmented_df, 198 are non-time varying


,LAT,LON,E_UNEMP,E_PCI,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,...,"Automatic_applications_sent_for_mail-in_ballots_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)",Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),Mention_of_tribal_casinos,Jan_1_2019_Minimum_Wage,Mar_29_2019_Minimum_Wage,Jul_1_2019_Minimum_Wage,Oct_1_2019_Minimum_Wage,Different_Minimum_Wage_for_Smaller_Businesses,positiveScore,hhs_region
fips,,,,,,,,,,,,,,,,,,,,,
1001.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1003.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,1.196777e-07,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1005.0,0.000000e+00,9.669865e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,4.273517e-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1007.0,4.825253e-07,1.364788e-06,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,4.264961e-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1009.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,1.701735e-07,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69120.0,2.551651e-07,2.041321e-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78010.0,2.421115e-07,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
78020.0,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [9]:
take_col = [name for name in non_time_varying_columns if any(take in name for take in ['take'])]
take_col

[]

### K-means

In [10]:
# Define Hyperparameters
X = hhs_augmented_df[non_time_varying_columns]#.to_dask_array(lengths=True)
y = np.log(hhs_augmented_df["rolled_cases"])#.to_dask_array(lengths=True)
n_clusters_range = range(1, 11)
window_size_range = range(2,15)



In [11]:
X

,LAT,LON,E_UNEMP,E_PCI,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_NOHSDP,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_MOBILE,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,EP_UNINSUR,MP_UNINSUR,State_FIPS_Code,Closed_businesses_overnight,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Face_mask_mandate_enforced_by_fines,Face_mask_mandate_enforced_by_criminal_charge/citation,No_legal_enforcement_of_face_mask_mandate,Liquor_stores_remained_open,Firearm_sellers_remained_open,Cannabis_dispensaries_considered_essential_business,Initially_reopen_restaurants_for_outdoor_dining_only,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_People_who_are_Incarcerated,Vaccine_allocation_phase:_Residents_of_Homeless_Shelters,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Vaccine_allocation_phase:_General_Public,Adults_ages_65+_prioritized_ahead_of_essential_workers,Prioritization_by_race/ethnicity,Proof_of_work_eligibility_requirement_for_vaccination,Proof_of_age_eligibility_requirement_for_vaccination,Proof_of_residency_requirement_for_vaccination,Penalty_for_failure_to_comply_with_vaccine_distribution_requirements,Expanded_scope_of_practice_of_certain_health_providers_to_administer_vaccines,State_previously_allowed_audio-only_telehealth,State_had_CHIP_premium_non-payment_lock-out_period_as_of_January_2019,Suspend_CHIP_premium_non-payment_lock-outs,Report_COVID-19_testing_by_race/ethnicity,Report_COVID-19_cases_by_race/ethnicity,Report_COVID-19_hospitalizations_by_race,Report_COVID-19_deaths_by_race/ethnicity,Report_COVID-19_vaccinations_by_race/ethnicity,Report_Indigenous_COVID-19_testing,Report_Indigenous_COVID-19_cases,Report_Indigenous_COVID-19_hospitalizations,Report_Indigenous_COVID-19_deaths,Report_Indigenous_COVID-19_vaccinations,Does_not_charge_copays_for_incarcerated_individuals,Waived_COVID/respiratory_illness-related_copays_during_pandemic_for_incarcerated_individuals,Waived_all_copays_during_pandemic_for_incarcerated_individuals,Did_not_waive_copays_for_incarcerated_individuals,No_state_unemployment_waiting_period_prior_to_pandemic;_or_date_waiting_period_waived_not_found,Reinstated_one_week_waiting_period_for_unemployment_insurance,Waived_work_search_requirement_for_unemployment_insurance,Reinstated_work_search_requirement_for_UI,Expanded_eligibility_of_unemployment_insurance_to_anyone_who_is_quarantined_and/or_taking_care_of_someone_who_is_quarantined,Expanded_eligibility_to_high-risk_individuals_in_preventative_quarantine,Expanded_eligibility_of_unemploym

In [14]:
@dask.delayed
def log_linear_regression(data):
    data = data.sort_values('days_from_start')
    n_obs = len(data)
    X = np.column_stack([data['days_from_start'], np.log(data[outcome_variable])])
    X = sm.add_constant(X)
    y = np.log(data[outcome_variable])
    results = []
    for i in range(window_size, n_obs):
        model = sm.OLS(y[i-window_size:i], X[i-window_size:i, :]).fit()
        results.append(model.params)
    return results

def cluster_kmeans(X_train, y_train, X_test, y_test, n_clusters):
    X_train = X_train.groupby('fips').first().reset_index().to_dask_array(lengths=True)
    y_train = y_train.groupby('fips').first().reset_index().to_dask_array(lengths=True)
    
    kmeans_model = dask_cluster.KMeans(n_clusters=n_clusters, random_state=0, init_max_iter=5)

    
    kmeans = kmeans_model.fit(X_train)
    labels_train = kmeans.predict(X_train)
    labels_test = kmeans.predict(X_test)
    
    X_train['cluster'] = labels_train
    X_test['cluster'] = labels_test
    
    return X_train, X_test

def run_experiment(X_train, y_train, X_test, y_test, n_clusters, window_size):
    # Perform clustering and regression
    labels_train, labels_test = cluster_kmeans(X_train, y_train, X_test, y_test, n_clusters)


    results_train = df_train.groupby('cluster').apply(log_linear_regression, meta=('results', 'f8'))
    results_train = results_train.compute()
    results_train = dd.from_pandas(pd.concat(results_train.values.tolist()), npartitions=1)

    results_test = df_test.groupby('cluster').apply(log_linear_regression, meta=('results', 'f8'))
    results_test = results_test.compute()
    results_test = dd.from_pandas(pd.concat(results_test.values.tolist()), npartitions=1)

    # Evaluate performance
    mse_train = mean_squared_error(y_train, X_train.merge(results_train, on=['cluster', 'days_from_start']).apply(lambda row: np.exp(np.dot(row[1:].values, row.index[1:].values)), axis=1))
    mse_test = mean_squared_error(y_test, X_test.merge(results_test, on=['cluster', 'days_from_start']).apply(lambda row: np.exp(np.dot(row[1:].values, row.index[1:].values)), axis=1))
    r2_train = r2_score(y_train, X_train.merge(results_train, on=['cluster', 'days_from_start']).apply(lambda row: np.exp(np.dot(row[1:].values, row.index[1:].values)), axis=1))
    r2_test = r2_score(y_test, X_test.merge(results_test, on=['cluster', 'days_from_start']).apply(lambda row: np.exp(np.dot(row[1:].values, row.index[1:].values)), axis=1))

    return (n_clusters, window_size, mse_train, mse_test, r2_train, r2_test)


In [15]:
experiments = []
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


for n_clusters in n_clusters_range:
    exp = cluster_kmeans(X_train, y_train, X_test, y_test, n_clusters)
    experiments.append(exp)
    break
cluster_results = dask.compute(*experiments)


#results_df = pd.DataFrame(results, columns=['n_clusters', 'window_size', 'mse_train', 'mse_test', 'r2_train', 'r2_test'])

AttributeError: 'DataFrame' object has no attribute 'take'

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train_unique = X_train.groupby('fips').first().reset_index()
X_test_unique = X_test.groupby('fips').first().reset_index()

distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker
distributed.nanny - WARNING - Restarting worker


AttributeError: 'DataFrame' object has no attribute 'take'